In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

import ellc
from ellc import lc

# import edmcmc as edm
# import batman

In [ ]:
def get_star_rv(rv_call_return):
    """
    ellc.rv may return a single array or a tuple (star_rv, planet_rv).
    This helper returns the star RV as a numpy array either way.
    """
    if isinstance(rv_call_return, (tuple, list)):
        print("tuple")
        return np.asarray(rv_call_return[0])
    return np.asarray(rv_call_return)

In [ ]:
# -----------------------
# Data input
# -----------------------
csvfile = "../data/HD97658_2025Feb20.csv"

df = pd.read_csv(csvfile, comment='#')
# ensure columns exist (uses your provided column names)
for col in ("ccfjdsum", "ccfrvmod", "dvrms"):
    if col not in df.columns:
        raise ValueError(f"Input CSV missing required column: {col}")

time_obs = df['ccfjdsum'].values.astype(float)  # times in days
rv_data = df['ccfrvmod'].values.astype(float)   # observed RV in km/s
rv_err = df['dvrms'].values.astype(float)       # RV uncertainties in km/s

In [ ]:
r_1 = 0.04132231404
r_2 = 0.0011228161490683
incl = 89.45
a = 24.2
e = 0.05
# f_c and f_s as you requested (note the -10 deg per your line)
f_c = math.sqrt(e) * math.cos(math.radians(-10.0))
f_s = math.sqrt(e) * math.sin(math.radians(-10.0))
q = 0.0000291991764706
t0 = 2460726.88970225  # start time
period = 9.489
lambda_1 = -40

vsini_1 = 3.0          # km/s (needed to produce RM). Change to your preferred value.
shape_1 = "sphere"
shape_2 = "sphere"
sbratio = 0.0          # planet contributes negligible light
flux_weighted = True   # compute flux-weighted RV (enables RM)

In [ ]:
# -----------------------
# Compute model at observation times
# -----------------------
rv_call = ellc.rv(
    time_obs,
    t_zero=t0,
    q=q,
    vsini_1=vsini_1,
    radius_1=r_1,
    radius_2=r_2,
    incl=incl,
    lambda_1=lambda_1,
    sbratio=sbratio,
    a=a,
    f_c=f_c,
    f_s=f_s,
    period=period,
    shape_1=shape_1,
    shape_2=shape_2,
    flux_weighted=flux_weighted,
)

star_rv = rv_call[0]   # km/s

In [ ]:
# -----------------------
# Align model and data systemic velocity (simple median offset)
# -----------------------
sys_offset = np.median(rv_data) - np.median(star_rv)
star_rv += sys_offset

In [ ]:
# -----------------------
# Plotting: RV vs time (data + model)
# -----------------------
plt.figure(figsize=(10, 5))
plt.errorbar(time_obs, rv_data * 1e3, yerr=rv_err * 1e3, fmt='o', ms=5, c='k', label='data', zorder=5)
plt.plot(time_obs, star_rv * 1e3, '-', lw=1.5, alpha=0.8, c='red', label='ellc model')

plt.xlabel('Time (days)')
plt.ylabel('Radial velocity (m/s)')
plt.legend()
plt.tight_layout()
plt.show()